# Node Classification Using a Three-Layer Graph Convolutional Network

This educational notebook demonstrates the implementation and mathematical foundation of a Graph Convolutional Network (GCN) from scratch using PyTorch. We will use a synthetic toy dataset representing students in a course, predicting whether they Pass or Fail based on their features and study-group connections (graph structure).

## 1. Imports

In [ ]:
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Add source directory to path to use our modules
sys.path.insert(0, os.path.abspath('..'))
from src.graph_data import build_dataset, set_all_seeds
from src.normalization import symmetric_normalization, normalize_adjacency_torch
from src.gcn import GCN, MLP
from src.train import train_model
from src.evaluate import evaluate_model
from src.visualization import plot_graph, plot_adjacency_heatmap, plot_training_curves, plot_confusion_matrix

%matplotlib inline

## 2. Reproducibility
To ensure results are deterministic, we set the random seeds for Python, NumPy, and PyTorch.

In [ ]:
seed = 42
set_all_seeds(seed)

## 3. Generate Graph & Dataset
We generate a synthetic student graph. 
- **Nodes (200):** Students (110 Pass, 90 Fail)
- **Edges:** Study group connections (homophilic: students of the same class tend to connect)
- **Features (3):** Study hours, attendance, assignment score. The two classes overlap strongly, so features alone are only moderately informative (an MLP gets about 78%).
- **Labels (2):** 0 = Fail, 1 = Pass

In [ ]:
data = build_dataset(seed=seed)  # defaults: 200 students, feature_gap=0.25
G = data["graph"]
labels = data["labels"]
features = data["features"]

## 4. Inspect Graph
Let's visualize the graph. Red nodes are students who fail, green nodes are students who pass.

In [ ]:
plot_graph(G, labels, title="Synthetic Student Graph (True Labels)")
print("Graph Statistics:")
for k, v in data["stats"].items():
    print(f"  {k}: {v}")

## 5. Node Features
The feature matrix $X$ has dimensions $N \times F$ (200 nodes, 3 features).

In [ ]:
print(f"Feature matrix X shape: {features.shape}")
print(f"First 3 nodes' standardized features:\n{features[:3]}")

## 6. Adjacency Matrix
The adjacency matrix $A$ encodes the graph structure. Since the graph is undirected, $A$ is symmetric.

In [ ]:
A = data["adjacency"]
print(f"Adjacency matrix A shape: {A.shape}")
plot_adjacency_heatmap(A, title="Original Adjacency Matrix A")

## 7. Self-Loops & Degree Matrix
To ensure nodes retain their own features during aggregation, we add self loops: $\hat{A} = A + I$.
Then we compute the degree matrix $\hat{D}$ of the augmented graph.

In [ ]:
I = np.eye(A.shape[0])
A_hat = A + I
D_hat = np.diag(A_hat.sum(axis=1))

print("First 5 diagonal entries of A_hat:", np.diag(A_hat)[:5])
print("First 5 degrees (diagonal of D_hat):", np.diag(D_hat)[:5])

## 8. Normalization
We symmetrically normalize the adjacency matrix to prevent high-degree nodes from dominating the aggregation:
$\tilde{A} = \hat{D}^{-1/2} \hat{A} \hat{D}^{-1/2}$

In [ ]:
A_norm_np, _, _, _ = symmetric_normalization(A)
plot_adjacency_heatmap(A_norm_np, title="Normalized Adjacency Matrix \u00C3", cmap="YlOrRd")

## 9. Define GCN
We define a 3-layer GCN. The architecture is:
`Input(3) -> GCNLayer(3->8) -> ReLU -> GCNLayer(8->8) -> ReLU -> GCNLayer(8->2) -> Logits`

In [ ]:
model = GCN(n_features=3, n_hidden=8, n_classes=2, n_layers=3, dropout=0.5)
print(model)
print(f"Total parameters: {model.count_parameters()}")

## 10. Train Model
We train the model using cross-entropy loss on the logits and the Adam optimizer, computing the loss only on nodes in the training mask. Dropout (0.5) and weight decay regularize the model, and **early stopping** watches the validation loss: training stops once it has not improved for 30 epochs, and the weights from the best epoch are restored.

In [ ]:
# Use the normalized adjacency Ã = D̂^(-1/2) (A + I) D̂^(-1/2) from Section 8.
# NOT data["t_adjacency"]: that is the raw A, with no self-loops (each node
# would ignore its own features) and no degree normalization.
A_norm = normalize_adjacency_torch(A)
X = data["t_features"]
t_labels = data["t_labels"]

history = train_model(
    model, A_norm, X, t_labels, 
    data["t_train_mask"], data["t_val_mask"],
    lr=0.01, epochs=500, patience=30, weight_decay=5e-4,
    seed=seed, print_every=50
)

## 11. Visualize Training
Let's plot the training curves.

In [ ]:
plot_training_curves(history, title_prefix="GCN-3 ")

## 12. Evaluate Model
We evaluate the model on the unseen test nodes.

In [ ]:
test_metrics = evaluate_model(model, A_norm, X, t_labels, data["t_test_mask"])
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test F1 Score: {test_metrics['f1']:.4f}")

plot_confusion_matrix(np.array(test_metrics["confusion_matrix"]), title="Test Set Confusion Matrix")

## 13. Compare Models
Let's compare the GCN against an MLP baseline that doesn't use the graph structure.

In [ ]:
set_all_seeds(seed)  # same starting point as the experiment script
mlp_model = MLP(n_features=3, n_hidden=8, n_classes=2, dropout=0.5)
mlp_history = train_model(
    mlp_model, A_norm, X, t_labels,
    data["t_train_mask"], data["t_val_mask"],
    lr=0.01, epochs=500, patience=30, weight_decay=5e-4, seed=seed, verbose=False
)
mlp_metrics = evaluate_model(mlp_model, A_norm, X, t_labels, data["t_test_mask"])

print(f"MLP Test Accuracy: {mlp_metrics['accuracy']:.4f}")
print(f"GCN Test Accuracy: {test_metrics['accuracy']:.4f}")
print("\nThe features of the two classes overlap, so the MLP, which sees each student")
print("in isolation, is limited. The GCN also averages over each student's study group,")
print("and because most neighbours share the student's result, this adds information.")